# 📝 Práctica M07: Arquitectura de Datos para Credit Scoring

**Escenario:** Tienes un dataset con la información de los últimos solicitantes de préstamos. Los datos están en "bruto": hay fechas, categorías de texto y escalas numéricas muy diferentes. Tu misión es aplicar ingeniería de variables para dejar el dataset listo para entrar en la fase de entrenamiento de modelos.

**Ejercicio 0:** Ejecuta esta celda para generar los datos que vas a transformar.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Creamos un dataset con retos de ingeniería reales
data_raw = pd.DataFrame({
    'ID_Solicitud': [101, 102, 103, 104, 105],
    'Fecha_Solicitud': pd.to_datetime(['2023-01-10', '2023-02-15', '2023-03-20', '2023-04-05', '2023-05-12']),
    'Fecha_Nacimiento': pd.to_datetime(['1985-05-10', '1992-08-22', '1978-01-05', '2000-11-30', '1988-03-15']),
    'Tipo_Vivienda': ['Alquiler', 'Propia', 'Alquiler', 'Propia', 'Hipoteca'],
    'Nivel_Educacion': ['Secundaria', 'Universitario', 'Universitario', 'Primaria', 'Secundaria'],
    'Saldo_Cuenta': [1500, 45000, 2800, 500, 12000],
    'Peticion_Credito': [5000, 200000, 15000, 2000, 35000]
})
data_raw.head()

,ID_Solicitud,Fecha_Solicitud,Fecha_Nacimiento,Tipo_Vivienda,Nivel_Educacion,Saldo_Cuenta,Peticion_Credito
0,101,2023-01-10,1985-05-10,Alquiler,Secundaria,1500,5000
1,102,2023-02-15,1992-08-22,Propia,Universitario,45000,200000
2,103,2023-03-20,1978-01-05,Alquiler,Universitario,2800,15000
3,104,2023-04-05,2000-11-30,Propia,Primaria,500,2000
4,105,2023-05-12,1988-03-15,Hipoteca,Secundaria,12000,35000


## 🛠️ EJERCICIOS

** Ejercicio 1: Creación de Variables Temporales**

Las fechas por sí solas no sirven para un modelo.
1. Crea una variable llamada Edad_Solicitante calculando la diferencia en años entre la Fecha_Solicitud y la Fecha_Nacimiento.
2. Crea una variable llamada Mes_Solicitud extrayendo el mes de la fecha de solicitud (útil para detectar estacionalidad en riesgos).

In [ ]:
data_raw['Edad_Solicitante'] = (data_raw['Fecha_Solicitud'] - data_raw['Fecha_Nacimiento']).dt.days //365
#Puesto que en 'dt.' no hay parámetro de años lo que hacemos es coger los días y dividirlos entre 365
data_raw['Mes_Solicitud'] = data_raw['Fecha_Solicitud'].dt.month
data_raw.head()


,ID_Solicitud,Fecha_Solicitud,Fecha_Nacimiento,Tipo_Vivienda,Nivel_Educacion,Saldo_Cuenta,Peticion_Credito,Edad_Solicitante,Mes_Solicitud
0,101,2023-01-10,1985-05-10,Alquiler,Secundaria,1500,5000,37,1
1,102,2023-02-15,1992-08-22,Propia,Universitario,45000,200000,30,2
2,103,2023-03-20,1978-01-05,Alquiler,Universitario,2800,15000,45,3
3,104,2023-04-05,2000-11-30,Propia,Primaria,500,2000,22,4
4,105,2023-05-12,1988-03-15,Hipoteca,Secundaria,12000,35000,35,5


**Ejercicio 2: Encoding de Categorías**

- **Ordinal Encoding:** Transforma Nivel_Educacion a una escala numérica donde: Primaria=1, Secundaria=2, Universitario=3.
- **One-Hot Encoding:** Transforma Tipo_Vivienda en columnas binarias (dummies). Asegúrate de que las columnas originales desaparezcan.

In [ ]:
mapeo = {'Primaria' : 1, 'Secundaria' : 2, 'Universitario' : 3}
data_raw['Nivel_Educacion_Num'] = data_raw['Nivel_Educacion'].map(mapeo)
#Usamos método map para que asigne valores a la columna nueva que creamos según la columna 
#sobre la que queremos hacer el 'encoding'
pd.get_dummies(data_raw, columns = ['Tipo_Vivienda'], prefix = 'Vivienda')

,ID_Solicitud,Fecha_Solicitud,Fecha_Nacimiento,Nivel_Educacion,Saldo_Cuenta,Peticion_Credito,Edad_Solicitante,Mes_Solicitud,Nivel_Educacion_Num,Vivienda_Alquiler,Vivienda_Hipoteca,Vivienda_Propia
0,101,2023-01-10,1985-05-10,Secundaria,1500,5000,37,1,2,True,False,False
1,102,2023-02-15,1992-08-22,Universitario,45000,200000,30,2,3,False,False,True
2,103,2023-03-20,1978-01-05,Universitario,2800,15000,45,3,3,True,False,False
3,104,2023-04-05,2000-11-30,Primaria,500,2000,22,4,1,False,False,True
4,105,2023-05-12,1988-03-15,Secundaria,12000,35000,35,5,2,False,True,False


**Ejercicio 3: Ingeniería de Ratios de Negocio**

Crea una variable llamada Ratio_Cobertura. Este ratio debe ser el Saldo_Cuenta dividido por la Peticion_Credito.

**Pregunta:** ¿Qué indica un ratio cercano a 0 frente a uno cercano a 1 para un analista de riesgos?

In [18]:
data_raw['Ratio_Cobertura'] = (data_raw['Saldo_Cuenta'] / data_raw['Peticion_Credito'])
data_raw.head()

,ID_Solicitud,Fecha_Solicitud,Fecha_Nacimiento,Tipo_Vivienda,Nivel_Educacion,Saldo_Cuenta,Peticion_Credito,Edad_Solicitante,Mes_Solicitud,Nivel_Educacion_Num,Ratio_Cobertura
0,101,2023-01-10,1985-05-10,Alquiler,Secundaria,1500,5000,37,1,2,0.300000
1,102,2023-02-15,1992-08-22,Propia,Universitario,45000,200000,30,2,3,0.225000
2,103,2023-03-20,1978-01-05,Alquiler,Universitario,2800,15000,45,3,3,0.186667
3,104,2023-04-05,2000-11-30,Propia,Primaria,500,2000,22,4,1,0.250000
4,105,2023-05-12,1988-03-15,Hipoteca,Secundaria,12000,35000,35,5,2,0.342857


Un ratio cercano a cero es que se acerca a no ser capaz de cubrir la petición de crédito que ha realizado, lo que indica alta probabilidad de no devolverlo es decir, de impago. Cercano a uno significa que es capaz de cubrir la peticion pedida pues no ha hecho un prestamo que multiplique en varias veces su saldo en cuenta lo que indica que es más capaz de afrontar los pagos y terminar solventando la petición de crédito.

**Ejercicio 4: Escalamiento (Feature Scaling)**

El modelo podría sesgarse porque la Peticion_Credito tiene valores de miles, mientras que la Edad_Solicitante tiene valores de decenas.

Aplica un StandardScaler a las variables Saldo_Cuenta, Peticion_Credito y Edad_Solicitante.

In [6]:
scaler = StandardScaler()
cols_escalar = ['Saldo_Cuenta', 'Peticion_Credito', 'Edad_Solicitante']
#Creamos una array con las variables a transformar y les aplicamos el 'scaler'
data_raw[cols_escalar] = scaler.fit_transform(data_raw[cols_escalar])
#Aplicamos el escalado a las columnas/variables deseadas

**Ejercicio 5: Limpieza Final**

Elimina las columnas de fechas originales (Fecha_Solicitud, Fecha_Nacimiento) y el ID_Solicitud, ya que no aportan valor predictivo al modelo.

In [8]:
data_final = data_raw.drop(['Fecha_Solicitud','Fecha_Nacimiento','ID_Solicitud'], axis = 1)
print("--Dataset Listo para Machine Learning---")
print(data_final)

--Dataset Listo para Machine Learning---
  Nivel_Educacion  Saldo_Cuenta  Peticion_Credito  Edad_Solicitante  \
0      Secundaria     -0.645448         -0.617087          0.419602   
1   Universitario      1.939909          1.976274         -0.498278   
2   Universitario     -0.568184         -0.484094          1.468608   
3        Primaria     -0.704881         -0.656985         -1.547283   
4      Secundaria     -0.021396         -0.218108          0.157351   

   Mes_Solicitud  Nivel_Educacion_Num  Viv_Alquiler  Viv_Hipoteca  Viv_Propia  \
0              1                    2          True         False       False   
1              2                    3         False         False        True   
2              3                    3          True         False       False   
3              4                    1         False         False        True   
4              5                    2         False          True       False   

   Ratio_Cobertura  
0         0.300000  
1  

## ✅ SOLUCIONES

In [5]:
# 1. Variables Temporales
data_raw['Edad_Solicitante'] = (data_raw['Fecha_Solicitud'] - data_raw['Fecha_Nacimiento']).dt.days // 365
data_raw['Mes_Solicitud'] = data_raw['Fecha_Solicitud'].dt.month

# 2. Encoding
# Ordinal
edu_map = {'Primaria': 1, 'Secundaria': 2, 'Universitario': 3}
data_raw['Nivel_Educacion_Num'] = data_raw['Nivel_Educacion'].map(edu_map)

# One-Hot
data_raw = pd.get_dummies(data_raw, columns=['Tipo_Vivienda'], prefix='Viv')

# 3. Ratio de Negocio
data_raw['Ratio_Cobertura'] = data_raw['Saldo_Cuenta'] / data_raw['Peticion_Credito']

# 4. Escalamiento
scaler = StandardScaler()
cols_to_scale = ['Saldo_Cuenta', 'Peticion_Credito', 'Edad_Solicitante']
data_raw[cols_to_scale] = scaler.fit_transform(data_raw[cols_to_scale])

# 5. Limpieza Final
data_final = data_raw.drop(['Fecha_Solicitud', 'Fecha_Nacimiento', 'ID_Solicitud', 'Nivel_Educacion'], axis=1)

print("--- Dataset Final Listo para Machine Learning ---")
print(data_final.head())

--- Dataset Final Listo para Machine Learning ---
   Saldo_Cuenta  Peticion_Credito  Edad_Solicitante  Mes_Solicitud  \
0     -0.645448         -0.617087          0.419602              1   
1      1.939909          1.976274         -0.498278              2   
2     -0.568184         -0.484094          1.468608              3   
3     -0.704881         -0.656985         -1.547283              4   
4     -0.021396         -0.218108          0.157351              5   

   Nivel_Educacion_Num  Viv_Alquiler  Viv_Hipoteca  Viv_Propia  \
0                    2          True         False       False   
1                    3         False         False        True   
2                    3          True         False       False   
3                    1         False         False        True   
4                    2         False          True       False   

   Ratio_Cobertura  
0         0.300000  
1         0.225000  
2         0.186667  
3         0.250000  
4         0.342857  
